## Problem statement is the best algorithm to predict user ratings for films
* Load dataset
* Require library
* data inspection
* 


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/netflix-inc/netflix-prize-data/README
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_1.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_2.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_3.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_4.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/movie_titles.csv
/kaggle/input/datasets/netflix-inc/netflix-prize-data/probe.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/qualifying.txt


In [2]:
import pandas as pd, numpy as np, gc

In [3]:
base = '/kaggle/input/datasets/netflix-inc/netflix-prize-data/'

def load_netflix(n):
    raw = pd.read_csv(base + f'combined_data_{n}.txt', header=None,
                      names=['customer_id', 'rating', 'date'], dtype=str)

    is_movie = raw['rating'].isna().to_numpy()       # movie ID lines have no rating
    pos = np.flatnonzero(is_movie)                    # where each movie block starts
    movie_ids = raw['customer_id'].iloc[pos].str[:-1].astype('int16').to_numpy()
    counts = np.diff(np.append(pos, len(raw)))        # size of each block
    raw['movie_id'] = np.repeat(movie_ids, counts)    # copy ID onto each row

    raw = raw[~is_movie]                              # drop the "1:" lines
    out = pd.DataFrame({
        'movie_id':    raw['movie_id'].to_numpy(),
        'customer_id': raw['customer_id'].astype('int32').to_numpy(),
        'rating':      raw['rating'].astype('int8').to_numpy(),
        'date':        pd.to_datetime(raw['date'], format='%Y-%m-%d'),
    })
    del raw
    gc.collect()
    return out

df1 = load_netflix(1); print('df1', df1.shape)
df2 = load_netflix(2); print('df2', df2.shape)
df3 = load_netflix(3); print('df3', df3.shape)
df4 = load_netflix(4); print('df4', df4.shape)

df1 (24053764, 4)
df2 (26977591, 4)
df3 (22601629, 4)
df4 (26847523, 4)


In [4]:
df = pd.concat([df1, df2, df3, df4], ignore_index=True)

In [5]:
print(df.shape)

(100480507, 4)


In [6]:
df.head()

,movie_id,customer_id,rating,date
0,1,1488844,3,2005-09-06
1,1,822109,5,2005-05-13
2,1,885013,4,2005-10-19
3,1,30878,4,2005-12-26
4,1,823519,3,2004-05-03


In [7]:
df.isnull().sum()

movie_id       0
customer_id    0
rating         0
date           0
dtype: int64